In [1]:
from pyexpat import EXPAT_VERSION
import socket
import struct
import time
import json
import joblib
import numpy as np
from pathlib import Path

# === Config ===
ROBOT_IP = 'localhost'
ROBOT_PORT = 18871
FEEDBACK_BYTES = 37 * 8
EXP_NAME = 'Oct30'
SEGMENT_NUMBER = 1
OUTPUT_FILE = '../data/' + EXP_NAME + '/2_DyF_L' + str(SEGMENT_NUMBER) + '.jsonl'
MODEL_PATH = '../models/' + EXP_NAME + '/1_Quasi_Static_Poly.pkl'

# === Load full-model (predicts FF_EE[0:6])
model = joblib.load(MODEL_PATH)
print(f"Loaded force + torque model from {MODEL_PATH}")

# === Connect to robot ===
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect((ROBOT_IP, ROBOT_PORT))
print(f"Connected to robot at {ROBOT_IP}:{ROBOT_PORT}")

# === Setup output directory
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)
log_file = open(OUTPUT_FILE, 'w')

# === Logging state
prev_time = time.time()
start_window_time = prev_time
msg_count = 0

# === Main loop
try:
    while True:
        data = sock.recv(FEEDBACK_BYTES)
        if not data:
            print("Connection closed by Simulink")
            break

        curr_time = time.time()
        delta_time = curr_time - prev_time
        prev_time = curr_time

        feedback = struct.unpack('<37d', data)

        POS = list(feedback[0:6])
        FRaw = list(feedback[6:12])
        FF_EE = list(feedback[12:18])
        FF_GB = list(feedback[18:24])
        Foot_Pedal = feedback[24]
        X_Dot_Cmds = list(feedback[25:31])
        X_DoubleDot_Cmds = list(feedback[31:37])

        # === Model prediction ===
        x_input = np.array([[POS[3], POS[4], POS[5]]])  # Only orientation
        ff_ee_pred = model.predict(x_input)[0]  # Predicts full FF_EE[0:6]

        # === Residual (dynamic component)
        ff_ee_dynamic = [measured - pred for measured, pred in zip(FF_EE, ff_ee_pred)]

        # === Log entry ===
        log_entry = {
            "timestamp": curr_time,
            "delta_time": delta_time,
            "POS": POS,
            "FRaw": FRaw,
            "FF_EE": FF_EE,
            "FF_EE_Quasi": list(ff_ee_pred),
            "FF_EE_Dynamic": ff_ee_dynamic,
            "FF_GB": FF_GB,
            "Foot_Pedal": Foot_Pedal,
            "X_Dot_Cmds": X_Dot_Cmds,
            "X_DoubleDot_Cmds": X_DoubleDot_Cmds
        }

        log_file.write(json.dumps(log_entry) + '\n')

        msg_count += 1
        if (curr_time - start_window_time) >= 0.5:
            rate = msg_count / (curr_time - start_window_time)
            print(f"[{curr_time:.2f}] {msg_count} msgs | Rate: {rate:.1f} Hz")
            start_window_time = curr_time
            msg_count = 0

except KeyboardInterrupt:
    print("Logging interrupted by user.")

finally:
    sock.close()
    log_file.close()
    print(f"Log saved to {OUTPUT_FILE}")


Loaded force + torque model from ../models/Oct30/1_Quasi_Static_Poly.pkl
Connected to robot at localhost:18871
[1761844081.52] 127 msgs | Rate: 252.4 Hz
[1761844082.02] 125 msgs | Rate: 249.7 Hz
[1761844082.52] 126 msgs | Rate: 250.1 Hz
[1761844083.03] 126 msgs | Rate: 250.0 Hz
[1761844083.53] 125 msgs | Rate: 249.9 Hz
[1761844084.03] 126 msgs | Rate: 250.4 Hz
[1761844084.53] 125 msgs | Rate: 249.8 Hz
[1761844085.03] 125 msgs | Rate: 249.9 Hz
[1761844085.54] 126 msgs | Rate: 250.1 Hz
[1761844086.04] 125 msgs | Rate: 249.8 Hz
[1761844086.54] 126 msgs | Rate: 250.1 Hz
[1761844087.04] 126 msgs | Rate: 250.2 Hz
[1761844087.55] 126 msgs | Rate: 250.1 Hz
[1761844088.05] 125 msgs | Rate: 249.9 Hz
[1761844088.55] 126 msgs | Rate: 250.1 Hz
[1761844089.05] 125 msgs | Rate: 249.9 Hz
[1761844089.56] 126 msgs | Rate: 250.1 Hz
[1761844090.06] 125 msgs | Rate: 249.9 Hz
[1761844090.56] 125 msgs | Rate: 250.0 Hz
[1761844091.06] 125 msgs | Rate: 249.7 Hz
[1761844091.56] 126 msgs | Rate: 250.2 Hz
[176184